In [0]:
from pyspark.sql import functions as F

LANDING = "/Volumes/workspace/climate_raw/landing"
BRONZE = "workspace.climate_bronze"


def ingest(folder, fmt, table, **options):
    """Incrementally load new files from a landing folder into a Bronze Delta table."""
    reader = (spark.readStream.format("cloudFiles")
              .option("cloudFiles.format", fmt)
              .option("cloudFiles.schemaLocation", f"{LANDING}/_schemas/{folder}"))
    for key, value in options.items():
        reader = reader.option(key, value)

    query = (reader.load(f"{LANDING}/{folder}/")
             .withColumn("_source_file", F.col("_metadata.file_path"))
             .withColumn("_ingested_at", F.current_timestamp())
             .writeStream
             .option("checkpointLocation", f"{LANDING}/_checkpoints/{folder}")
             .trigger(availableNow=True)          # process all new files, then stop
             .toTable(f"{BRONZE}.{table}"))
    query.awaitTermination()
    print(f"{table}: done")

In [0]:
ingest("districts", "csv", "districts",     header="true")
ingest("weather",   "csv", "weather_daily", header="true")
ingest("soil",      "csv", "soil_raw",      header="true")

In [0]:
for t in ["districts", "weather_daily", "soil_raw"]:
    n = spark.table(f"{BRONZE}.{t}").count()
    print(f"{t:14} {n:>10,} rows")

display(spark.sql(f"""
    SELECT COUNT(DISTINCT district_id)  AS districts,
           COUNT(DISTINCT _source_file) AS files,
           MIN(_ingested_at)            AS first_load,
           MAX(_ingested_at)            AS last_load
    FROM {BRONZE}.weather_daily
"""))

In [0]:
# RESET: uncomment and run ONLY to reload everything from scratch
# for folder, table in [("districts", "districts"), ("weather", "weather_daily"), ("soil", "soil_raw")]:
#     spark.sql(f"DROP TABLE IF EXISTS {BRONZE}.{table}")
#     dbutils.fs.rm(f"{LANDING}/_checkpoints/{folder}", True)
#     dbutils.fs.rm(f"{LANDING}/_schemas/{folder}", True)